# ResNet50 Baseline

Generates prediction CSVs using a pretrained ResNet50 (no training). Use this as a reference baseline.

## Configuration

In [1]:
DATASET_ROOT = "../datasets/dataset_a"
OUTPUT       = "../predictions/dataset_a.csv"
BATCH_SIZE   = 64
NUM_WORKERS  = 0  # >0 causes multiprocessing errors in Jupyter on macOS

import torch
# Auto-detects: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

In [2]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm

## Dataset

In [3]:
def load_required_parquet(root, parquet_file, required_columns):
    parquet_path = os.path.join(root, parquet_file)
    if not os.path.exists(parquet_path):
        raise FileNotFoundError(f"Missing parquet file: {parquet_path}")

    df = pd.read_parquet(parquet_path)
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{parquet_file} is missing required columns: {missing}")
    return df


def resolve_image_path(root, image_path):
    rel = str(image_path).replace("\\", "/").lstrip("./")
    candidates = [
        os.path.join(root, rel),
        os.path.join(root, "images", rel),
        os.path.join(root, "images", os.path.basename(rel)),
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]


class FaceDataset(Dataset):
    def __init__(self, root, image_size=(112, 112)):
        self.root = root
        df = load_required_parquet(root, "test.parquet", ["image_path", "template_id", "media_id"])
        self.image_paths = df["image_path"].tolist()
        self.template_ids = df["template_id"].values
        self.media_ids = df["media_id"].values
        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = resolve_image_path(self.root, self.image_paths[idx])
        if not os.path.exists(path):
            raise FileNotFoundError(f"Image not found: {path}. Check image_path values in parquet and dataset root.")
        image = Image.open(path).convert("RGB")
        return self.transform(image), idx

## Model

In [4]:
class ResNetEncoder(nn.Module):
    def __init__(self, device="cuda"):
        super().__init__()
        self.device = device
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Identity()
        self.to(device).eval()

    @torch.inference_mode()
    def encode(self, images):
        return F.normalize(self.backbone(images.to(self.device)), p=2, dim=1)

## Template Aggregation

In [5]:
def aggregate_template_features(embeddings, template_ids, media_ids):
    template_features = {}
    for tid in np.unique(template_ids):
        mask = template_ids == tid
        t_embs = embeddings[mask]
        t_mids = media_ids[mask]
        media_feats = []
        for mid in np.unique(t_mids):
            media_feats.append(t_embs[t_mids == mid].mean(axis=0))
        feat = np.sum(media_feats, axis=0)
        feat = feat / (np.linalg.norm(feat) + 1e-8)
        template_features[tid] = feat
    return template_features

## Run Baseline

In [6]:
device = DEVICE if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

dataset = FaceDataset(DATASET_ROOT)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))
pairs_df = load_required_parquet(DATASET_ROOT, "pairs.parquet", ["template_id_1", "template_id_2"])
print(f"Images: {len(dataset)}, Pairs: {len(pairs_df)}")

model = ResNetEncoder(device)
embeddings_list, indices_list = [], []
for images, indices in tqdm(dataloader, desc="Encoding"):
    embeddings_list.append(model.encode(images).cpu().numpy())
    indices_list.append(indices.numpy())

embeddings = np.vstack(embeddings_list)
indices = np.concatenate(indices_list)
embeddings = embeddings[np.argsort(indices)]

print("Aggregating template features...")
template_features = aggregate_template_features(embeddings, dataset.template_ids, dataset.media_ids)

print("Computing pair scores...")
tid_list = sorted(template_features.keys())
tid_to_idx = {tid: i for i, tid in enumerate(tid_list)}
feat_matrix = np.zeros((len(tid_list), embeddings.shape[1]), dtype=np.float32)
for tid, feat in template_features.items():
    feat_matrix[tid_to_idx[tid]] = feat

t1s = pairs_df["template_id_1"].values
t2s = pairs_df["template_id_2"].values
scores = np.zeros(len(pairs_df), dtype=np.float32)
batch = 500_000
for i in tqdm(range(0, len(pairs_df), batch), desc="Scoring"):
    end = min(i + batch, len(pairs_df))
    idx1 = np.array([tid_to_idx[t] for t in t1s[i:end]])
    idx2 = np.array([tid_to_idx[t] for t in t2s[i:end]])
    scores[i:end] = np.sum(feat_matrix[idx1] * feat_matrix[idx2], axis=1)

Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"template_id_1": t1s, "template_id_2": t2s, "score": scores}).to_csv(OUTPUT, index=False)
print(f"Predictions saved to: {OUTPUT} ({len(scores)} pairs)")

Device: cpu
Images: 227630, Pairs: 8010270


Encoding:  13%|███▋                        | 475/3557 [12:57<1:24:04,  1.64s/it]


KeyboardInterrupt: 